### Lahore Air Quality & Smog Forecast
#### Notebook 5: AQI Conversion & Forecast Generation
Purpose:

This notebook converts the predicted PM2.5 concentrations generated by the machine learning models into Air Quality Index (AQI) values. It then assigns AQI categories, smog risk levels, and health recommendations, producing forecasts that can be displayed on the website in a clear and user-friendly format.

In [16]:
import pandas as pd
import numpy as np
import joblib

In [17]:
df = pd.read_csv("lahore_air_quality_features.csv")

In [18]:
df["date"] = pd.to_datetime(df["date"])
df.head()

,name,date,pm25,temperature,relativehumidity,wind_speed,wind_direction,lat,lon,location_id,is_smog_season,pm25_day1,pm25_day2,pm25_day3
0,"ARC, Lahore",2025-11-10,267.000000,17.669231,66.076923,1.215385,147.000000,31.522056,74.314944,6125629,1,267.708333,265.611111,234.470588
1,"ARC, Lahore",2025-11-11,267.708333,18.341667,64.875000,3.308333,208.625000,31.522056,74.314944,6125629,1,265.611111,234.470588,322.333333
2,"ARC, Lahore",2025-11-12,265.611111,16.227778,68.888889,3.638889,296.000000,31.522056,74.314944,6125629,1,234.470588,322.333333,270.125000
3,"ARC, Lahore",2025-11-13,234.470588,16.564706,64.764706,3.605882,293.764706,31.522056,74.314944,6125629,1,322.333333,270.125000,231.878261
4,"ARC, Lahore",2025-11-14,322.333333,15.166667,73.866667,1.386667,192.466667,31.522056,74.314944,6125629,1,270.125000,231.878261,283.727273


In [19]:
model_day1 = joblib.load("pm25_day1_model.pkl")
model_day2 = joblib.load("pm25_day2_model.pkl")
model_day3 = joblib.load("pm25_day3_model.pkl")

In [20]:
latest = (
    df.sort_values("date")
      .groupby("name")
      .tail(1)
      .reset_index(drop=True)
)
latest.head()

,name,date,pm25,temperature,relativehumidity,wind_speed,wind_direction,lat,lon,location_id,is_smog_season,pm25_day1,pm25_day2,pm25_day3
0,US Diplomatic Post: Lahore,2025-02-15,110.833333,18.125000,57.458333,3.870833,142.708333,31.560078,74.335890,8664,1,142.083333,164.250000,222.166667
1,"Assistant Commissioner Office, Allama Iqbal",2025-07-26,0.319952,31.212500,78.666667,3.004167,113.583333,31.471880,74.239512,4608421,0,0.296889,0.026300,0.445000
2,"The Bank of Punjab, Head Office, BOP Tower, Gu...",2025-11-20,124.500000,21.650000,57.000000,3.650000,20.500000,31.508051,74.337544,6135466,1,202.500000,81.100000,139.000000
3,Climate Finance Pakistan (PakAirQuality),2025-12-06,183.273333,13.406667,68.666667,4.833333,293.933333,31.537800,74.344100,6135373,1,201.118182,151.526316,168.225000
4,SOS Johar Town LHR,2025-12-07,152.594737,15.121053,67.842105,3.494737,313.368421,31.454400,74.282500,6135496,1,109.128571,122.150000,96.500000


In [21]:
features = [
    "pm25",
    "temperature",
    "relativehumidity",
    "wind_speed",
    "wind_direction"
]
X = latest[features]

In [22]:
latest["pm25_day1"] = model_day1.predict(X)
latest["pm25_day2"] = model_day2.predict(X)
latest["pm25_day3"] = model_day3.predict(X)

In [23]:
def pm25_to_aqi(pm):

    breakpoints = [
        (0.0,12.0,0,50),
        (12.1,35.4,51,100),
        (35.5,55.4,101,150),
        (55.5,150.4,151,200),
        (150.5,250.4,201,300),
        (250.5,350.4,301,400),
        (350.5,500.4,401,500)
    ]

    for c_low,c_high,aqi_low,aqi_high in breakpoints:

        if c_low <= pm <= c_high:

            return round(
                ((aqi_high-aqi_low)/(c_high-c_low))
                *(pm-c_low)
                +aqi_low
            )
    return 500

In [24]:
latest["AQI Day 1"] = latest["pm25_day1"].apply(pm25_to_aqi)
latest["AQI Day 2"] = latest["pm25_day2"].apply(pm25_to_aqi)
latest["AQI Day 3"] = latest["pm25_day3"].apply(pm25_to_aqi)

In [25]:
def aqi_category(aqi):

    if aqi <= 50:
        return "Good"
    elif aqi <=100:
        return "Moderate"
    elif aqi <=150:
        return "Unhealthy for Sensitive Groups"
    elif aqi <=200:
        return "Unhealthy"
    elif aqi <=300:
        return "Very Unhealthy"
    else:
        return "Hazardous"

In [26]:
def smog_risk(aqi):

    if aqi <=50:
        return "Low"
    elif aqi <=100:
        return "Mild"
    elif aqi <=150:
        return "Moderate"
    elif aqi <=200:
        return "High"
    elif aqi <=300:
        return "Very High"
    else:
        return "Extreme"

In [27]:
def health_advice(aqi):

    if aqi <=50:
        return "Enjoy outdoor activities."
    elif aqi <=100:
        return "Sensitive individuals should monitor symptoms."
    elif aqi <=150:
        return "Sensitive groups should reduce prolonged outdoor activity."
    elif aqi <=200:
        return "Everyone should limit prolonged outdoor exertion."
    elif aqi <=300:
        return "Avoid outdoor activities whenever possible."
    else:
        return "Remain indoors and wear a high-quality mask if going outside."

In [28]:
for day in ["AQI Day 1","AQI Day 2","AQI Day 3"]:

    latest[f"{day} Category"] = latest[day].apply(aqi_category)
    latest[f"{day} Risk"] = latest[day].apply(smog_risk)
    latest[f"{day} Advice"] = latest[day].apply(health_advice)

In [29]:
forecast = latest[[
    "name",
    "AQI Day 1",
    "AQI Day 1 Category",
    "AQI Day 1 Risk",
    "AQI Day 1 Advice",

    "AQI Day 2",
    "AQI Day 2 Category",
    "AQI Day 2 Risk",
    "AQI Day 2 Advice",

    "AQI Day 3",
    "AQI Day 3 Category",
    "AQI Day 3 Risk",
    "AQI Day 3 Advice"
]]
forecast.head(20)

,name,AQI Day 1,AQI Day 1 Category,AQI Day 1 Risk,AQI Day 1 Advice,AQI Day 2,AQI Day 2 Category,AQI Day 2 Risk,AQI Day 2 Advice,AQI Day 3,AQI Day 3 Category,AQI Day 3 Risk,AQI Day 3 Advice
0,US Diplomatic Post: Lahore,183,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,187,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,190,Unhealthy,High,Everyone should limit prolonged outdoor exertion.
1,"Assistant Commissioner Office, Allama Iqbal",6,Good,Low,Enjoy outdoor activities.,10,Good,Low,Enjoy outdoor activities.,48,Good,Low,Enjoy outdoor activities.
2,"The Bank of Punjab, Head Office, BOP Tower, Gu...",266,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.,192,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,229,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.
3,Climate Finance Pakistan (PakAirQuality),235,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.,220,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.,220,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.
4,SOS Johar Town LHR,184,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,197,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,181,Unhealthy,High,Everyone should limit prolonged outdoor exertion.
5,"BAM, Civil Secretariat",322,Hazardous,Extreme,Remain indoors and wear a high-quality mask if...,391,Hazardous,Extreme,Remain indoors and wear a high-quality mask if...,174,Unhealthy,High,Everyone should limit prolonged outdoor exertion.
6,"Abdullah Khokhar, House Johar Town",217,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.,193,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,238,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.
7,Johar Town,220,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.,218,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.,224,Very Unhealthy,Very High,Avoid outdoor activities whenever possible.
8,University of Management and Technology,155,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,165,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,170,Unhealthy,High,Everyone should limit prolonged outdoor exertion.
9,FF Pakistan,181,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,175,Unhealthy,High,Everyone should limit prolonged outdoor exertion.,187,Unhealthy,High,Everyone should limit prolonged outdoor exertion.


In [30]:
forecast.to_csv("lahore_forecast.csv", index=False)

### Reflection

In this notebook, I transformed the predicted PM2.5 concentrations from the machine learning models into Air Quality Index (AQI) values using the U.S. EPA AQI breakpoints. Each forecast was then classified into an AQI category, assigned a smog risk level, and paired with a health recommendation. The resulting forecast table provides clear, user-friendly information that can be integrated into the project website to help communicate air quality conditions and encourage informed public health decisions.